# Key Evidence ↔ Syscall 证据匹配（以 CoinMiner 为例）

本笔记本用于解析 `malware_report` 中各类别报告，从每个 report 的 **Key Evidence** 段落提取多行证据，再到对应 `raw` 目录中按 `sha256` 查找 syscall 日志，检索是否能找到匹配证据。示例以 **CoinMiner** 类别演示，可切换 `FAMILY` 参数扩展到其他类别。

## 1. 加载并索引 raw/syscall 数据（按 sha256）

扫描 `raw` 文件夹，将文件名中的 `sha256` 映射到对应的 syscall 日志文件路径，避免一次性加载全部内容。

In [21]:
from pathlib import Path
import re
import json
from typing import Dict, List, Tuple

ROOT = Path("/home/zzh/Compass-Proj1/MirrorShield/ELF_samples")
FAMILY = "CoinMiner"
RAW_DIR = ROOT / "malware_report" / FAMILY / "raw"
REPORT_ROOT = ROOT / "malware_report" / FAMILY / "analysis_reports_comparison"


def build_raw_index(raw_dir: Path) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = {}
    if not raw_dir.is_dir():
        print(f"[WARN] raw 目录不存在: {raw_dir}")
        return index
    for p in raw_dir.glob("*.jsonl"):
        # 文件名格式: <sha>.elf_YYYY-...jsonl
        m = re.match(r"([0-9a-f]{64})\.elf_", p.name)
        if not m:
            continue
        sha = m.group(1)
        index.setdefault(sha, []).append(p)
    return index

raw_index = build_raw_index(RAW_DIR)
print(f"raw 索引数量: {len(raw_index)}")
print("示例 sha:", list(raw_index.keys())[:3])

raw 索引数量: 36
示例 sha: ['f0dc05f4a1ab641186035edcf039b2cc2b301cdeedd041463093501605a9969c', 'ed50d381726b6b765130c2f6da39be2ce85fbcb829ae2444902b258a94f7feb8', '895f8dff9cd26424b691a401c92fa7745e693275c38caf6a6aff277eadf2a70b']


## 2. 解析 malware_report（按类别与样本）

读取报告文件，解析出 `sha256` 与原始 report 文本。

In [22]:
def list_report_files(report_root: Path) -> List[Path]:
    if not report_root.is_dir():
        print(f"[WARN] report 目录不存在: {report_root}")
        return []
    files: List[Path] = []
    for bucket in sorted(p for p in report_root.iterdir() if p.is_dir()):
        for rpt in bucket.glob("*_deepseek_report.txt"):
            files.append(rpt)
    return files


def parse_report_text(path: Path) -> Tuple[str, str]:
    text = path.read_text(errors="ignore")
    # 优先从 Target 字段提取 sha256
    m = re.search(r"Target:\s*`([0-9a-fA-F]{64})(?:\.elf)?`", text)
    sha = m.group(1).lower() if m else ""
    # 兜底：从全文中抓取第一个 64 位 hex
    if not sha:
        m2 = re.search(r"([0-9a-fA-F]{64})", text)
        sha = m2.group(1).lower() if m2 else ""
    return sha, text

report_files = list_report_files(REPORT_ROOT)
print(f"报告数量: {len(report_files)}")
print("示例文件:", report_files[:3])

报告数量: 35
示例文件: [PosixPath('/home/zzh/Compass-Proj1/MirrorShield/ELF_samples/malware_report/CoinMiner/analysis_reports_comparison/01_MALICIOUS/707302c6_deepseek_report.txt'), PosixPath('/home/zzh/Compass-Proj1/MirrorShield/ELF_samples/malware_report/CoinMiner/analysis_reports_comparison/01_MALICIOUS/5f1f8499_deepseek_report.txt'), PosixPath('/home/zzh/Compass-Proj1/MirrorShield/ELF_samples/malware_report/CoinMiner/analysis_reports_comparison/01_MALICIOUS/0213dc20_deepseek_report.txt')]


## 3. 提取每个 report 的 Key Evidence 行

从报告中定位 `### 🔍 Key Evidence` 段落，提取项目符号行作为待匹配证据。

In [ ]:
def extract_key_evidence(text: str) -> List[str]:
    lines = text.splitlines()
    start = None
    for i, ln in enumerate(lines):
        if "Key Evidence" in ln:
            start = i + 1
            break
    if start is None:
        return []

    evidences: List[str] = []
    for ln in lines[start:]:
        if ln.startswith("### ") or ln.startswith("## "):
            break
        ln = ln.strip()
        if ln.startswith("-"):
            ln = ln.lstrip("- ")
            if ln:
                evidences.append(ln)
    return evidences

## 4. 在对应 sha256 的 syscall 中匹配证据（以 CoinMiner 为例）

为每条 Key Evidence 提取关键词，并在对应 `raw/*.jsonl` 中检索匹配行。

In [ ]:
KEYWORD_PATTERN = re.compile(r"[A-Z_]{3,}|/[^\s]+")


def extract_keywords(evidence: str) -> List[str]:
    kws = KEYWORD_PATTERN.findall(evidence)
    # 兜底：补充一些常见动作词
    if not kws:
        kws = re.findall(r"[a-zA-Z]{4,}", evidence)
    # 统一大小写（路径保留原样，其他转小写）
    norm = []
    for k in kws:
        if k.startswith("/"):
            norm.append(k)
        else:
            norm.append(k.lower())
    return list(dict.fromkeys(norm))


def find_raw_files_for_sha(sha: str, index: Dict[str, List[Path]]) -> List[Path]:
    return index.get(sha, [])


def search_jsonl_for_keywords(jsonl_path: Path, keywords: List[str], limit: int = 5) -> List[str]:
    matches: List[str] = []
    if not jsonl_path.is_file() or not keywords:
        return matches
    kw_lower = [k.lower() if not k.startswith("/") else k for k in keywords]
    try:
        with jsonl_path.open() as f:
            for line in f:
                low = line.lower()
                if any((k in low) if not k.startswith("/") else (k in line) for k in kw_lower):
                    matches.append(line.strip())
                    if len(matches) >= limit:
                        break
    except Exception as exc:
        print(f"[WARN] 读取失败 {jsonl_path}: {exc}")
    return matches


def match_evidence_to_syscalls(sha: str, evidences: List[str], index: Dict[str, List[Path]], per_evidence_limit: int = 3):
    raw_files = find_raw_files_for_sha(sha, index)
    if not raw_files:
        return {"raw_files": [], "matches": {}}
    jsonl_path = raw_files[0]
    matches = {}
    for ev in evidences:
        kws = extract_keywords(ev)
        lines = search_jsonl_for_keywords(jsonl_path, kws, limit=per_evidence_limit)
        matches[ev] = {"keywords": kws, "lines": lines}
    return {"raw_files": raw_files, "matches": matches}

In [ ]:
# 选择一个样本进行演示（优先从 01_MALICIOUS 里取）
first_report = None
for p in report_files:
    if "01_MALICIOUS" in str(p):
        first_report = p
        break
if first_report is None and report_files:
    first_report = report_files[0]

if first_report is None:
    print("[WARN] 未找到报告文件")
else:
    sha, text = parse_report_text(first_report)
    evidences = extract_key_evidence(text)
    print("Report:", first_report)
    print("SHA:", sha)
    print("Key Evidence 行数:", len(evidences))
    for ev in evidences:
        print(" -", ev)

    if not sha:
        print("[WARN] 未解析到 sha，无法进行 raw 对比")
    else:
        result = match_evidence_to_syscalls(sha, evidences, raw_index, per_evidence_limit=2)
        print("\nRaw files:", result["raw_files"])
        for ev, info in result["matches"].items():
            print("\nEvidence:", ev)
            print("Keywords:", info["keywords"])
            for line in info["lines"]:
                print("  ", line)

## 4.1 提取 C2 服务器 IP（可选）

从 raw/jsonl 中扫描可能的 IP 地址（IPv4/IPv6），用于辅助识别潜在 C2。

In [ ]:
IPV4_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
IPV6_RE = re.compile(r"\b(?:[A-Fa-f0-9]{0,4}:){2,7}[A-Fa-f0-9]{0,4}\b")


def extract_ips_from_line(line: str) -> List[str]:
    ips = IPV4_RE.findall(line) + IPV6_RE.findall(line)
    # 过滤明显无效的 IPv4（如 999.999.999.999）
    valid = []
    for ip in ips:
        if "." in ip:
            parts = ip.split(".")
            if all(p.isdigit() and 0 <= int(p) <= 255 for p in parts):
                valid.append(ip)
        else:
            valid.append(ip)
    return valid


def extract_c2_ips_from_jsonl(jsonl_path: Path, limit: int = 50) -> List[str]:
    ips: List[str] = []
    if not jsonl_path.is_file():
        return ips
    try:
        with jsonl_path.open() as f:
            for line in f:
                # 直接扫描文本
                ips.extend(extract_ips_from_line(line))
                if len(ips) >= limit:
                    break
    except Exception as exc:
        print(f"[WARN] 读取失败 {jsonl_path}: {exc}")
    # 去重保持顺序
    seen = set()
    out = []
    for ip in ips:
        if ip not in seen:
            seen.add(ip)
            out.append(ip)
    return out

# 演示：从上一个样本的 raw 文件中提取 IP
if first_report is not None:
    sha, _ = parse_report_text(first_report)
    raw_files = find_raw_files_for_sha(sha, raw_index)
    if raw_files:
        ips = extract_c2_ips_from_jsonl(raw_files[0], limit=30)
        print("SHA:", sha)
        print("C2/IP 候选:", ips)
    else:
        print("[WARN] 未找到对应 raw 文件")

SHA: b39bff6adb3947886e8689be98aca658b276362a2df330d7952222ccd52a209e
C2/IP 候选: ['10.0.0.1']


## 5. 汇总匹配结果并导出

对多个样本做批量匹配，生成摘要表，并导出 CSV/JSON。

In [ ]:
def summarize_reports(files: List[Path], index: Dict[str, List[Path]], limit: int = 5):
    rows = []
    for rpt in files[:limit]:
        sha, text = parse_report_text(rpt)
        if not sha:
            continue
        evidences = extract_key_evidence(text)
        match = match_evidence_to_syscalls(sha, evidences, index, per_evidence_limit=1)
        matched_lines = sum(len(v["lines"]) for v in match["matches"].values())
        raw_files = find_raw_files_for_sha(sha, index)
        ips = extract_c2_ips_from_jsonl(raw_files[0], limit=10) if raw_files else []
        rows.append({
            "sha": sha,
            "report": rpt.name,
            "evidence_count": len(evidences),
            "matched_lines_count": matched_lines,
            "c2_ip_count": len(ips),
            "c2_ips": ";".join(ips),
        })
    return rows

summary_rows = summarize_reports(report_files, raw_index, limit=5)
summary_rows

[{'sha': 'b39bff6adb3947886e8689be98aca658b276362a2df330d7952222ccd52a209e',
  'report': '707302c6_deepseek_report.txt',
  'evidence_count': 6,
  'matched_lines_count': 3,
  'c2_ip_count': 1,
  'c2_ips': '10.0.0.1'},
 {'sha': '5a5f75838de210ddf51489b5aed2736ffbac9930a7a2b98ad6d6216be01f89f5',
  'report': '5f1f8499_deepseek_report.txt',
  'evidence_count': 5,
  'matched_lines_count': 5,
  'c2_ip_count': 0,
  'c2_ips': ''},
 {'sha': '9876b83ce63edd0a9df541fef95407d95c87a124c4801206c176fe2ed5ed461c',
  'report': '0213dc20_deepseek_report.txt',
  'evidence_count': 8,
  'matched_lines_count': 6,
  'c2_ip_count': 0,
  'c2_ips': ''},
 {'sha': '2ba1775bbb0f5abc6c89f0334cb42588cbd2a5c6903995912611649607c07d9b',
  'report': 'fc2302e1_deepseek_report.txt',
  'evidence_count': 8,
  'matched_lines_count': 7,
  'c2_ip_count': 0,
  'c2_ips': ''},
 {'sha': '895f8dff9cd26424b691a401c92fa7745e693275c38caf6a6aff277eadf2a70b',
  'report': '69d4e00a_deepseek_report.txt',
  'evidence_count': 6,
  'matched_l

In [28]:
import csv

def export_summary(rows, out_csv: Path, out_json: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else [])
        if rows:
            writer.writeheader()
            writer.writerows(rows)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    with out_json.open("w") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

out_csv = ROOT / "coinminer_evidence_summary.csv"
out_json = ROOT / "coinminer_evidence_summary.json"
if summary_rows:
    export_summary(summary_rows, out_csv, out_json)
    print("已导出:", out_csv, out_json)
else:
    print("[WARN] 无可导出的摘要")

已导出: /home/zzh/Compass-Proj1/MirrorShield/ELF_samples/coinminer_evidence_summary.csv /home/zzh/Compass-Proj1/MirrorShield/ELF_samples/coinminer_evidence_summary.json
